In [20]:
import pandas as pd
from joblib import load
from moe_model import ScannerMixtureOfExpertsBootstrap
import numpy as np
import os

# Block 2: From Features to Classification

UX goal: take one feature row and return predicted probability plus final class based on the optimal threshold.

## Feature Source

The UX can support two scenarios:
- From radiologist segmentation (features generated in Block 1, standard mode)
- From automatic segmentation (features generated in Block 1)

Core constraint:
- Feature names and order must match those used during training.

## Request Input Data

The UI should provide:
- Subject identifier
- Ready feature table (single row)
- Automatic check for missing or extra columns

Recommended UX fallback:
- If columns are misaligned, block inference and show a diff between expected and received columns.

In [11]:

path = 'SNIPPET_from_aligned_images_to_features'
subj = 'PRISM 001'


path = '../SNIPPET_from_aligned_images_to_features'

im_path = path + 'data/' + subj + '/' + 'images' + '/' + np.array(os.listdir(f'{path}/data/{subj}/images')).astype(str)
appo = []
for im in im_path:
    if 'nii.gz' in im:
        appo.append(im)
im_path = sorted(appo)


segm_path = path + 'data/' + subj + '/' + 'labels' + '/' + np.array(os.listdir(f'{path}/data/{subj}/labels')).astype(str)

appo = []
for im in segm_path:
    if 'nii.gz' in im:
        appo.append(im)
        
        
segm_path = sorted(appo)
df = pd.DataFrame(np.append(im_path, segm_path))

df = df.transpose().rename(columns = {0: 'images_ADC', 1: 'images_T2', 2: 'segm'})

df['Subject']= subj

subj_df = pd.read_csv(f'{path}/data/{df["Subject"].iloc[0]}/features/df.csv')

In [12]:
subj_df.iloc[0]

T2_glrlm_RunEntropy                             2.180344
T2_gldm_DependenceEntropy                       3.435755
T2_gldm_DependenceVariance                      2.548204
T2_glszm_ZonePercentage                         0.217391
ADC_gldm_SmallDependenceLowGrayLevelEmphasis    0.047165
T2_gldm_DependenceNonUniformityNormalized       0.179584
ADC_glcm_SumEntropy                             1.518331
T2_gldm_SmallDependenceEmphasis                 0.167585
psa_density                                     0.122364
Name: 0, dtype: float64

# INPUT DATA FROM AUTOMATIC SEGMENTATIONS

In [22]:
from pathlib import Path
import pandas as pd

path = Path("../SNIPPET_automatic_segmentation")

subjs_aligned = sorted([p.name for p in path.iterdir() if p.is_dir()])
subj = subjs_aligned[0]

# images
images = sorted((path / subj / "images").glob("*.nii.gz"))

adc_path = [p for p in images if "_0000" in p.name][0]
t2_path  = [p for p in images if "_0001" in p.name][0]

# segmentations
segm_paths = sorted((path / subj / "labels").glob("*.nii.gz"))

# clinical
psa_path = path / subj / "features" / "clinical" / "psa_density.csv"
psa_density = pd.read_csv(psa_path)["psa_density"].iloc[0]

# final dataframe: one row per segmentation
df = pd.DataFrame({
    "Subject": subj,
    "images_T2": str(t2_path),
    "images_ADC": str(adc_path),
    "segm": [str(p) for p in segm_paths],
    "psa_density": psa_density,
})

df

,Subject,images_T2,images_ADC,segm,psa_density
0,ext0000,..\automatic_segmentation\ext0000\images\ext00...,..\automatic_segmentation\ext0000\images\ext00...,..\automatic_segmentation\ext0000\labels\ext00...,0.122364
1,ext0000,..\automatic_segmentation\ext0000\images\ext00...,..\automatic_segmentation\ext0000\images\ext00...,..\automatic_segmentation\ext0000\labels\ext00...,0.122364


In [23]:
df_automatic_segm = pd.read_csv(f"{path}/{df['Subject'].iloc[0]}/features/df.csv")

# LOAD THE CLASSIFIER

In [24]:
bundle = load("data_report_GT/moe_classifier_trained.joblib")

model_loaded = bundle["model"]
threshold_summary_loaded = bundle["threshold_summary"]
feature_names_loaded = bundle["feature_names"]

c:\Users\michele.vodret00\AppData\Local\miniconda3\envs\tf-gpu-env\lib\site-packages\sklearn\base.py:376: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.7.2 when using version 1.5.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\michele.vodret00\AppData\Local\miniconda3\envs\tf-gpu-env\lib\site-packages\sklearn\base.py:376: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.7.2 when using version 1.5.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\michele.vodret00\AppData\Local\miniconda3\envs\tf-gpu-env\lib\site-packages\sklearn\base.py:376: InconsistentVe

In [25]:
optimal_threshold_corrected = threshold_summary_loaded['median_threshold_f2_cv']

# Classification 

Outputs to display in the interface:
- Predicted probability (0-1)
- Operating threshold
- Final class: GG <= 2 or GG >= 3

UX suggestion:
- Always show probability and threshold together, not only the class, for decision transparency.

## Result

In [ ]:
##### FROM RADIOLOGIST SEGMENTATION

I = 0
case = subj_df.iloc[[I]]
pred_proba = model_loaded.predict_proba(case)[:,1]

print("radiologist segmentation case")
print(f"Predicted probability for sample: {pred_proba[I]:.4f} VS Optimal threshold: {optimal_threshold_corrected:.4f}")

if pred_proba[I] >= optimal_threshold_corrected:
    print('predicted class is GG >= 3')

else:
    print('predicted class is GG < 3')

Predicted probability for sample: 0.1281 VS Optimal threshold: 0.3092
predicted class is GG < 3


In [30]:

## FROM AUTOMATIC SEGMENTATION

for J in range(len(df_automatic_segm)):

    case_automatic_segm = df_automatic_segm.iloc[[J]]

    pred_proba = model_loaded.predict_proba(case_automatic_segm)[:,1]


    print("automatic segmentation case")

    print(f"Predicted probability for sample: {pred_proba[I]:.4f} VS Optimal threshold: {optimal_threshold_corrected:.4f}")

    if pred_proba[I] >= optimal_threshold_corrected:
        print('predicted class is GG >= 3')

    else:
        print('predicted class is GG < 3')

automatic segmentation case
Predicted probability for sample: 0.3045 VS Optimal threshold: 0.3092
predicted class is GG < 3
automatic segmentation case
Predicted probability for sample: 0.1787 VS Optimal threshold: 0.3092
predicted class is GG < 3
